# RS/OF fit in different layers

We have a few fit of the RS/OF data for the treadmill, let's check how do they compare
across the layers

%load_ext autoreload
%autoreload 2

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import flexiznam as flz
import pandas as pd
import numpy as np
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import seaborn as sns
from pathlib import Path

# Style
sns.set_theme(style="ticks", context="talk")
font_path = '/nemo/lab/znamenskiyp/home/shared/resources/fonts/arial.ttf'
fm.fontManager.addfont(font_path)
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = 'Arial'
import matplotlib as mpl
mpl.rcParams['pdf.fonttype'] = 42   # TrueType, editable text in Illustrator
mpl.rcParams['ps.fonttype'] = 42    # same for EPS/PS output

SESSIONS_TO_EXCLUDE = {
    "PZAG22.1b_S20260220": "1000 more frames than triggers in the treadmill recording"
}


In [ ]:
# Load neurons_df with trial-averaged fits
import pandas as pd

pickle_path = "/camp/home/blota/code/v1_depth_map/neurons_df_trial_average.pickle"
neurons_df = pd.read_pickle(pickle_path)


print(f"Total: {len(neurons_df)} neurons from {neurons_df['session'].nunique()} sessions.")

In [ ]:
bad_sess = neurons_df[neurons_df.session=='PZAG22.1b_S20260306b']
bad_roi = bad_sess[bad_sess['rsof_test_rsq_closedloop_g2d_treadmill'] < -600000]
print(bad_roi.roi)

In [ ]:
bad_sess = neurons_df[neurons_df.session=='PZAG22.1b_S20260306b']
bad_roi = bad_sess[bad_sess['rsof_test_rsq_closedloop_g2d_treadmill_trial_average'] < -600000]
print(bad_roi.roi)

In [ ]:
# Adding nominal depth info
import numpy as np
layer_cutoff = 350
PROJECTS = ["ccyp_l5_3d_vision", "colasa_3d-vision_revisions"]
# 1. Fetch all session metadata for the projects
project_sessions_list = []

for project in PROJECTS:
    flexilims_session = flz.get_flexilims_session(project_id=project)
    sessions = flz.get_entities("session", flexilims_session=flexilims_session)
    project_sessions_list.append(sessions)

all_sessions_df = pd.concat(project_sessions_list)

# Ensure the index is the session name for mapping
if 'name' in all_sessions_df.columns:
    all_sessions_df = all_sessions_df.set_index('name')

# 2. Map the nominal_depth from the session metadata to the neurons_df
neurons_df['nominal_depth'] = neurons_df['session'].map(all_sessions_df['nominal_depth'])

# 3. Handle edge cases where nominal_depth might be stored as an array/list for a single session
neurons_df['nominal_depth'] = neurons_df['nominal_depth'].apply(
    lambda x: np.mean(x) if isinstance(x, (list, np.ndarray, pd.Series)) else x
)
neurons_df['layer'] =  neurons_df['nominal_depth'].apply(
    lambda x: 'Layer 2/3' if x < layer_cutoff else 'Layer 5')

print(f"Added nominal_depth. Range: {neurons_df['nominal_depth'].min()} - {neurons_df['nominal_depth'].max()}")


In [ ]:
[c for c in neurons_df.columns if 'crossval' in c and 'gadd' in c]

In [ ]:
# Concatenate all sessions for closedloop rsof
models = ["gof_treadmill", "grs_treadmill", "gadd_treadmill", "g2d_treadmill", "gratio_treadmill", "g2mult_treadmill", "gprod_treadmill"]
model_labels = ["OF only", "RS only", "Additive", "Conjunctive", "Pure depth", 'Depth x RS', 'Product']

models = ["gof_treadmill", "grs_treadmill", "g2d_treadmill", "gratio_treadmill", "g2mult_treadmill"]
model_labels = ["OF only", "RS only",  "Conjunctive", "Pure depth", 'Depth x RS']

cols = [
    "roi",
    "best_depth",
    "preferred_depth_closedloop",
    "preferred_depth_closedloop_crossval",
    "depth_tuning_popt_closedloop",
    "depth_tuning_test_rsq_closedloop",
    "depth_tuning_test_spearmanr_pval_closedloop",
    "depth_tuning_test_spearmanr_rval_closedloop",
    "preferred_RS_closedloop_g2d",
    "preferred_RS_closedloop_crossval_g2d",
    "preferred_OF_closedloop_g2d",
    "preferred_OF_closedloop_crossval_g2d",
]
cols_to_add = [
    "rsof_test_rsq_closedloop_",
    "rsof_rsq_closedloop_",
    "rsof_popt_closedloop_",
]
for model in models:
    for col in cols_to_add:
        cols.append(f"{col}{model}_trial_average")


In [ ]:
model_cols = [f"rsof_test_rsq_closedloop_{model}_trial_average" for model in models]
# Find the best model for each neuron
neurons_df["best_model"] = neurons_df[model_cols].idxmax(axis=1)




## Empirical null threshold for RS/OF R²

Fit a Gaussian to the negative tail of each cross-validated R² distribution. Because the null (no tuning) should be symmetric around 0, the negative half is an uncontaminated sample of it. Mirror it to estimate σ, then threshold at the desired percentile.

In [ ]:
from scipy.stats import norm

def empirical_null_threshold(rsq_values, percentile=95):
    """
    Estimate the null R² distribution from the negative tail and return a threshold.

    Assumes the null is N(0, sigma): fit sigma from the negative values (which are
    uncontaminated by real signal), then return the given percentile of N(0, sigma).
    """
    vals = np.asarray(rsq_values, dtype=float)
    vals = vals[np.isfinite(vals)]
    neg = vals[vals < 0]
    if len(neg) < 20:
        raise ValueError(f"Only {len(neg)} negative values — too few to fit null reliably")
    # sigma of the symmetric null: std of the mirrored distribution equals std of neg values
    sigma = np.std(neg)
    threshold = norm.ppf(percentile / 100, loc=0, scale=sigma)
    return threshold, sigma


PERCENTILE = 95


# Column shortcuts used throughout the notebook
rs_col    = 'rsof_test_rsq_closedloop_grs_treadmill_trial_average'
of_col    = 'rsof_test_rsq_closedloop_gof_treadmill_trial_average'
depth_col = 'rsof_test_rsq_closedloop_gratio_treadmill_trial_average'


In [ ]:
# Apply empirical null threshold to every RS/OF model (trial-average columns only)
model_rsq_cols = {
    model: f'rsof_test_rsq_closedloop_{model}_trial_average'
    for model in models
}

thresholds = {}
for model, col in model_rsq_cols.items():
    vals = neurons_df[col].dropna().values
    vals = vals[np.isfinite(vals)]
    vals = vals[vals >= -1]   # drop sentinel values (e.g. -10000) that inflate sigma
    thr, sigma = empirical_null_threshold(vals, percentile=PERCENTILE)
    thresholds[model] = thr
    sig_col = f'{col}_sig'
    neurons_df[sig_col] = neurons_df[col] > thr
    print(f"{model:30s}  sigma={sigma:.4f}  threshold={thr:.4f}  "
          f"passing={neurons_df[sig_col].mean()*100:.1f}%")

# Convenience aliases used elsewhere in the notebook
rs_threshold = thresholds['grs_treadmill']
of_threshold = thresholds['gof_treadmill']
neurons_df['rs_sig']   = neurons_df[f"{model_rsq_cols['grs_treadmill']}_sig"]
neurons_df['of_sig']   = neurons_df[f"{model_rsq_cols['gof_treadmill']}_sig"]
neurons_df['rsof_sig'] = neurons_df['rs_sig'] | neurons_df['of_sig']
neurons_df['any_fig_sig'] = neurons_df[[f"{rsq_col}_sig" for rsq_col in model_rsq_cols.values()]].any(axis=1)
print(f"Total neurons: {len(neurons_df)} | RS sig: {neurons_df['rs_sig'].sum()} | OF sig: {neurons_df['of_sig'].sum()} | RS or OF sig: {neurons_df['rsof_sig'].sum()} | Any model sig: {neurons_df['any_fig_sig'].sum()}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, col, label in zip(
    axes,
    [rs_col, of_col],
    ['RS', 'OF'],
):
    vals = neurons_df[col].dropna().values
    vals = vals[np.isfinite(vals)]

    threshold, sigma = empirical_null_threshold(vals, percentile=PERCENTILE)
    print(f"{label}: sigma={sigma:.4f}, threshold (p{PERCENTILE})={threshold:.4f}")

    # Histogram of full distribution
    counts, _, _ = ax.hist(vals, bins=200, range=(-0.5, 1.0), density=True,
                           color='steelblue', alpha=0.6, label='All neurons')

    # Scale Gaussian peak to match histogram peak
    scale = counts.max() / norm.pdf(0, 0, sigma)
    x = np.linspace(-0.6, 1.0, 500)
    ax.plot(x, norm.pdf(x, 0, sigma) * scale, color='tomato', lw=2,
            label=f'Null N(0, {sigma:.3f})')
    ax.axvline(threshold, color='tomato', ls='--', lw=2,
               label=f'p{PERCENTILE} threshold = {threshold:.3f}')

    frac_above = np.mean(vals > threshold)
    ax.set_xlabel(f'{label} $r^2$')
    ax.set_ylabel('Density')
    ax.set_xlim(-0.5, 1.0)
    ax.set_title(f'{label}: {frac_above*100:.1f}% above threshold')
    ax.legend(fontsize=8)

sns.despine()
plt.tight_layout()

In [ ]:
fig =plt.figure(figsize=(10,4))
neurons_df["preferred_depth_amplitude"] = neurons_df[
    "depth_tuning_popt_closedloop_treadmill"
].apply(lambda x: np.exp(x[0]) + x[-1])
neurons_df_sig = neurons_df[
     (neurons_df["depth_tuning_test_spearmanr_rval_closedloop_treadmill"] > 0.1)
     & (neurons_df["depth_tuning_test_spearmanr_pval_closedloop_treadmill"] < 0.05)
    # (neurons_df["preferred_depth_amplitude"] >0.2)
     & neurons_df['any_fig_sig']
]

axes_list = []
for iax, layer in enumerate(['Layer 2/3', 'Layer 5']):

    if iax == 0:
        ax = fig.add_subplot(1, 2, 1)
    else:
        ax = fig.add_subplot(1, 2, 2, sharey=axes_list[0])
    axes_list.append(ax)
    markersize=8
    alpha=0.7
    color='k'
    fontsize_dict={"title": 21, "label": 18, "tick": 18}
    ci=None


    # Calculate percentage of neurons that have the best model
    subset_df = neurons_df_sig[neurons_df_sig.layer == layer]
    neuron_sum = (
        subset_df.groupby("session")[["roi"]].agg(["count"]).values.flatten()
    )
    props = []
    # calculate the proportion of neurons that have the best model for each session
    for i, model in enumerate(model_cols):
        prop = (
            subset_df.groupby("session")[["best_model", "roi"]]
            .apply(lambda x: x[x["best_model"] == model][["roi"]].agg(["count"]))
            .values.flatten()
        ) / neuron_sum
        props.append(prop)
        # Plot bar plot
    for i, model in enumerate(model_cols):
        sns.stripplot(
            x=np.ones(len(props[i])) * i,
            y=props[i],
            size=markersize,
            alpha=alpha,
            jitter=0.12,
            edgecolor="white",
            color=sns.color_palette("Set1")[i],
        )
        plt.plot(
            [i - 0.3, i + 0.3],
            [np.median(props[i]), np.median(props[i])],
            linewidth=3,
            color=color,
        )
        if ci is not None:
            plt.fill_between(
                [i - 0.3, i + 0.3],
                [ci[i][0], ci[i][0]],
                [ci[i][1], ci[i][1]],
                color=sns.color_palette("dark")[i],
                alpha=0.7,
                edgecolor="none",
            )

    sns.despine(offset=5, ax=plt.gca())
    ax.set_xticks(range(len(models)))
    ax.set_xticklabels(model_labels, fontsize=fontsize_dict["label"], rotation=90)
    ax.tick_params(axis='y', labelsize=fontsize_dict["label"])
    if iax == 0:
        ax.set_ylabel(
            "Proportion of neurons with best model fit",
            fontsize=fontsize_dict["label"],
        )
    else:
        ax.tick_params(labelleft=False)
    ax.set_ylim([0, 0.6])
    ax.set_title(layer)


In [ ]:
fig =plt.figure(figsize=(10,5))

neurons_df["preferred_depth_amplitude"] = neurons_df[
    "depth_tuning_popt_closedloop_treadmill"
].apply(lambda x: np.exp(x[0]) + x[-1])
neurons_df_sig = neurons_df[
    (neurons_df["depth_tuning_test_spearmanr_rval_closedloop_treadmill"] > 0.1)
    & (neurons_df["depth_tuning_test_spearmanr_pval_closedloop_treadmill"] < 0.05)
]
for iax, layer in enumerate(['Layer 2/3', 'Layer 5']):

    ax = fig.add_subplot(1,2,1 + iax)
    markersize=10
    alpha=0.7
    color='k'
    fontsize_dict={"title": 15, "label": 10, "tick": 10}
    ci=None


    # Calculate percentage of neurons that have the best model
    subset_df = neurons_df_sig[neurons_df_sig.layer == layer]
    neuron_sum = (
        subset_df.groupby("session")[["roi"]].agg(["count"]).values.flatten()
    )
    props = []
    # calculate the proportion of neurons that have the best model for each session
    for i, model in enumerate(model_cols):
        prop = (
            subset_df.groupby("session")[["best_model", "roi"]]
            .apply(lambda x: x[x["best_model"] == model][["roi"]].agg(["count"]))
            .values.flatten()
        ) / neuron_sum
        props.append(prop)
        # Plot bar plot
    for i, model in enumerate(model_cols):
        sns.stripplot(
            x=np.ones(len(props[i])) * i,
            y=props[i],
            size=markersize,
            alpha=alpha,
            jitter=0.4,
            edgecolor="white",
            color=sns.color_palette("dark")[i],
        )
        plt.plot(
            [i - 0.4, i + 0.4],
            [np.median(props[i]), np.median(props[i])],
            linewidth=3,
            color=color,
        )
        if ci is not None:
            plt.fill_between(
                [i - 0.4, i + 0.4],
                [ci[i][0], ci[i][0]],
                [ci[i][1], ci[i][1]],
                color=sns.color_palette("dark")[i],
                alpha=0.7,
                edgecolor="none",
            )

    sns.despine(offset=5, ax=plt.gca())
    ax.set_xticks(range(len(models)))
    ax.set_xticklabels(model_labels, fontsize=fontsize_dict["label"], rotation=90)
    ax.tick_params(axis='y', labelsize=fontsize_dict["label"])
    ax.set_ylabel(
        "Proportion of neurons with best model fit",
        fontsize=fontsize_dict["label"],
    )
    ax.set_ylim([0, 0.5])
    ax.set_title(layer)


In [ ]:
neurons_df["preferred_depth_amplitude"] = neurons_df[
    "depth_tuning_popt_closedloop_treadmill"
].apply(lambda x: np.exp(x[0]) + x[-1])
neurons_df_sig = neurons_df[
    (neurons_df["depth_tuning_test_spearmanr_rval_closedloop_treadmill"] > 0.1)
    & (neurons_df["depth_tuning_test_spearmanr_pval_closedloop_treadmill"] < 0.05)
]

In [ ]:
fig =plt.figure(figsize=(15,5))

ax0 = fig.add_subplot(1,2,1)
ax1 = fig.add_subplot(1,2,2)
temp_df = []
for iax, layer in enumerate(['Layer 2/3', 'Layer 5']):
    # Calculate percentage of neurons that have the best model
    subset_df = neurons_df_sig[neurons_df_sig.layer == layer]
    g2d= subset_df['rsof_test_rsq_closedloop_g2d_treadmill_trial_average'].values
    gratio = subset_df['rsof_test_rsq_closedloop_g2mult_treadmill_trial_average'].values
    valid = (g2d > 0) & (gratio >0)
    df = pd.DataFrame(dict(g2d=g2d, gratio=gratio, layer=layer, session=subset_df.session.values, diff_conjratio=g2d/gratio))
    bad = pd.isna(df.g2d) | pd.isna(df.gratio)
    temp_df.append(df[~bad])
temp_df = pd.concat(temp_df, ignore_index=True) 
temp_df['diff_conjratio'] = np.log2(np.clip(temp_df['diff_conjratio'], 0.1e-5, 1e3))
_ = sns.histplot(temp_df, x='diff_conjratio', hue='layer',stat='density', common_norm=False, binrange=(-4,6),  binwidth=0.2, element="step", cumulative=False, ax=ax0, kde=False)
for l, d in temp_df.groupby('layer'):
    ax0.scatter(np.nanmean(d.diff_conjratio), 0.6, marker='v', color='C0' if l == 'Layer 2/3' else 'C1')
_ = sns.histplot(temp_df, x='diff_conjratio', hue='layer',stat='density', common_norm=False, binrange=(-10,10), binwidth=1e-3, element="step", cumulative=True, ax=ax1)
ax1.axhline(1, color='grey', ls='--', zorder=-2)


fold_changes = [1/10, 1/2, 1, 2, 10, 100]
fold_labels = ["1/10", "1/2", "1", "2", "10", "100"]
tick_positions = np.log2(fold_changes)
for ax in [ax0, ax1]:
    ax.set_xlabel('Conjunctive $R^2$ vs Depth x RS $R^2$ fold change')
    # Set the locations of the ticks
    ax.set_xticks(tick_positions)
    # Set the text of the ticks
    ax.set_xticklabels(fold_labels)
ax0.set_xlim(-4,6)
ax1.set_xlim(-5,10)
l = ax0.legend()
l.set_visible(False)
plt.tight_layout()

In [ ]:
fig =plt.figure(figsize=(15,5))

ax0 = fig.add_subplot(1,2,1)
ax1 = fig.add_subplot(1,2,2)
temp_df = []
for iax, layer in enumerate(['Layer 2/3', 'Layer 5']):
    # Calculate percentage of neurons that have the best model
    subset_df = neurons_df_sig[neurons_df_sig.layer == layer]
    g2d= subset_df['rsof_test_rsq_closedloop_g2d_treadmill_trial_average'].values
    gratio = subset_df['rsof_test_rsq_closedloop_g2mult_treadmill_trial_average'].values
    
    df = pd.DataFrame(dict(g2d=g2d, gratio=gratio, layer=layer, session=subset_df.session.values, diff_conjratio=g2d-gratio))
    bad = pd.isna(df.g2d) | pd.isna(df.gratio)
    temp_df.append(df[(~bad)])
temp_df = pd.concat(temp_df, ignore_index=True) 
temp_df['diff_conjratio'] = np.clip(temp_df['diff_conjratio'], -1, 1)
_ = sns.histplot(temp_df, x='diff_conjratio',
                  hue='layer',stat='density', common_norm=False,
                    binrange=(-0.5,1),  binwidth=0.002, element="step", cumulative=False, ax=ax0, kde=False)
_ = sns.histplot(temp_df, x='diff_conjratio', hue='layer',stat='density',
                  common_norm=False, binrange=(-0.5,1), binwidth=1e-3, element="step", cumulative=True, ax=ax1)
ax1.axhline(1, color='grey', ls='--', zorder=-2)


for ax, h in zip([ax0, ax1], [40, 1.05]):
    for l, d in temp_df.groupby('layer'):
        ax.scatter(np.nanmean(d.diff_conjratio), h, marker='v', color='C0' if l == 'Layer 2/3' else 'C1', label=f'Mean {l}')
        ax.scatter(np.nanmedian(d.diff_conjratio), h, marker='|', color='C0' if l == 'Layer 2/3' else 'C1', label=f'Median {l}')

    ax.set_xlabel('Conjunctive $R^2$ - Depth x RS $R^2$')
ax1.legend(loc='lower right')
l = ax0.legend()
l.set_visible(False)
plt.tight_layout()

In [ ]:
fig =plt.figure(figsize=(15,5))

ax0 = fig.add_subplot(1,2,1)
ax1 = fig.add_subplot(1,2,2)
temp_df = []
for iax, layer in enumerate(['Layer 2/3', 'Layer 5']):
    # Calculate percentage of neurons that have the best model
    subset_df = neurons_df_sig[neurons_df_sig.layer == layer]
    g2mult= subset_df['rsof_test_rsq_closedloop_g2mult_treadmill_trial_average'].values
    gratio = subset_df['rsof_test_rsq_closedloop_gratio_treadmill_trial_average'].values
    df = pd.DataFrame(dict(g2d=g2mult, gratio=gratio, layer=layer, session=subset_df.session.values, diff_conjratio=g2mult-gratio))
    bad = pd.isna(df.g2d) | pd.isna(df.gratio)
    temp_df.append(df[~bad])
temp_df = pd.concat(temp_df, ignore_index=True) 
temp_df['diff_conjratio'] = np.clip(temp_df['diff_conjratio'], -1, 1)
_ = sns.histplot(temp_df, x='diff_conjratio', hue='layer',stat='density', common_norm=False, binrange=(-0.05,0.2),  binwidth=0.002, element="step", cumulative=False, ax=ax0, kde=False)
_ = sns.histplot(temp_df, x='diff_conjratio', hue='layer',stat='density', common_norm=False, binrange=(-0.05,0.2), binwidth=1e-3, element="step", cumulative=True, ax=ax1)
ax1.axhline(1, color='grey', ls='--', zorder=-2)


for ax, h in zip([ax0, ax1], [100, 1.05]):
    for l, d in temp_df.groupby('layer'):
        ax.scatter(np.nanmean(d.diff_conjratio), h, marker='v', color='C0' if l == 'Layer 2/3' else 'C1', label=f'Mean {l}')
        ax.scatter(np.nanmedian(d.diff_conjratio), h, marker='|', color='C0' if l == 'Layer 2/3' else 'C1', label=f'Median {l}')

    ax.set_xlabel('Depth x RS $R^2$ - Depth  $R^2$')
ax1.legend(loc='lower right')
l = ax0.legend()
l.set_visible(False)
plt.tight_layout()

In [ ]:

def plot_binned_avg(ax, x_series, y_series, color):
    """Groups x_series into bins and plots the mean +- SEM of y_series"""
    df = pd.DataFrame({'x': x_series, 'y': y_series}).dropna()
    df['bin'] = pd.cut(df['x'], bins=bins)
    
    # Calculate mean and Standard Error of the Mean (SEM)
    binned = df.groupby('bin', observed=False)['y']
    means = binned.mean()
    sems = binned.sem()
    
    # Get the center of each bin for the x-coordinates of our line plot
    centers = [b.mid for b in means.index]
    
    # Plot errorbars
    ax.errorbar(centers, means, yerr=sems, color=color, fmt='-o', 
                linewidth=3, capsize=5,markersize=1, zorder=5)


In [ ]:
from functools import partial

cl = partial(np.clip, a_min=-0.1, a_max=np.inf)
fig =plt.figure(figsize=(15,5))

ax0 = fig.add_subplot(1,2,1, aspect='equal')
ax1 = fig.add_subplot(1,2,2, aspect='equal')
color = ['C0' if c=='Layer 2/3' else 'C1' for c in neurons_df_sig.layer]


ax1.scatter(
    cl(neurons_df_sig.rsof_test_rsq_closedloop_g2mult_treadmill_trial_average), 
    cl(neurons_df_sig.rsof_test_rsq_closedloop_g2d_treadmill_trial_average), 
    c=color, alpha=0.3, s=30, edgecolors='None')
ax0.scatter(
    cl(neurons_df_sig.rsof_test_rsq_closedloop_gratio_treadmill_trial_average),
    cl(neurons_df_sig.rsof_test_rsq_closedloop_g2mult_treadmill_trial_average), 
    c=color, alpha=0.3, s=30, edgecolors='None')
for ax in [ax0, ax1]:
    ax.plot([-0.1,1],[-0.1,1],color='grey', ls='--', alpha=0.3, zorder=-1    )
    ax.set_xlim(-0.1, 1)
    ax.set_ylim(-0.1, 1)
    sns.despine()
# Define along the x-axis
bw = 0.1
bins = np.arange(-bw/2, 0.9+bw/2, bw)

# Loop through each layer to plot the binned averages on both axes
for layer_name, color_val in zip(['Layer 2/3', 'Layer 5'], ['C0', 'C1']):
    mask = neurons_df_sig.layer == layer_name
    
    # Data for ax1 (x = g2mult, y = g2d)
    x1 = cl(neurons_df_sig[mask].rsof_test_rsq_closedloop_g2mult_treadmill_trial_average)
    y1 = cl(neurons_df_sig[mask].rsof_test_rsq_closedloop_g2d_treadmill_trial_average)
    plot_binned_avg(ax1, x1, y1, color_val)
    
    # Data for ax0 (x = gratio, y = g2mult)
    x0 = cl(neurons_df_sig[mask].rsof_test_rsq_closedloop_gratio_treadmill_trial_average)
    y0 = cl(neurons_df_sig[mask].rsof_test_rsq_closedloop_g2mult_treadmill_trial_average)
    plot_binned_avg(ax0, x0, y0, color_val)


import matplotlib.lines as mlines
# Define the colors and labels
legend_elements = [
    mlines.Line2D([0], [0], marker='o', color='w', markerfacecolor='C0', markersize=8, label='Layer 2/3'),
    mlines.Line2D([0], [0], marker='o', color='w', markerfacecolor='C1', markersize=8, label='Layer 5')
]
ax1.legend(handles=legend_elements, loc='upper left', frameon=False)
ax1.set_xlabel('Depth x RS model $r^2$')
ax1.set_ylabel('Cunjonctive model $r^2$')
ax0.set_xlabel('Pure depth model $r^2$')
ax0.set_ylabel('Depth x RS model $r^2$')

In [ ]:
(neurons_df_sig.rsof_test_rsq_closedloop_g2mult_treadmill_trial_average <= -10000).sum()

In [ ]:
neurons_df["preferred_depth_amplitude"] = neurons_df[
    "depth_tuning_popt_closedloop_treadmill"
].apply(lambda x: np.exp(x[0]) + x[-1])
neurons_df_sig = neurons_df[
    (neurons_df["depth_tuning_test_spearmanr_rval_closedloop_treadmill"] > 0.1)
    & (neurons_df["depth_tuning_test_spearmanr_pval_closedloop_treadmill"] < 0.05)
]
print(len(neurons_df.groupby('session').value_counts(['layer'])))


In [ ]:
[sess for sess in neurons_df['session'].unique() if sess not in neurons_df_sig.session.unique()]
sdf = neurons_df[neurons_df.session == 'BRAC12025.1g_S20251212']
sdf.depth_tuning_test_spearmanr_rval_closedloop_treadmill

In [ ]:
# Group by both, compute the mean, and flatten the index
mod2use = [f"rsof_test_rsq_closedloop_{mod}" for mod in ["grs_treadmill_trial_average", "gof_treadmill_trial_average", 'gratio_treadmill_trial_average', 'g2mult_treadmill_trial_average', "gadd_treadmill_trial_average", "gprod_treadmill_trial_average", 'g2d_treadmill_trial_average']]

valid = neurons_df[[f"{mod}_sig" for mod in mod2use]].any(axis=1)
print(f"Valid neurons: {valid.sum()} / {len(neurons_df)} ({valid.mean()*100:.1f}%)")
avg_by_sess = neurons_df_sig[valid].groupby(['session', 'layer']).mean(numeric_only=True).reset_index()
fig, ax = plt.subplots(1,1, figsize=(5,5))
plot_models = mod2use
plot_labels = ['RS', 'OF', 'Depth', 'Depth x RS', 'Add', 'Product', 'Conjunctive']
for isess, series in avg_by_sess.iterrows():
    l = series.layer == 'Layer 2/3'
    ax.plot(
        np.arange(len(plot_models)) + float(l) * 0.15, 
        cl(series[plot_models]),
        mfc='C0' if l else 'C1',
        alpha=0.5,
        color='k',
        lw=0.5,
        marker='o')
ax.set_xticks(0.15/2+ np.arange(len(plot_models)), labels=plot_labels, rotation=90)
ax.set_ylabel('Model $r^2$')
sns.despine()

In [ ]:
avg_by_sess.groupby('layer')[plot_models].mean(numeric_only=True).T

In [ ]:
# Group by both, compute the mean, and flatten the index
neurons_df_sig = neurons_df[
    (neurons_df["depth_tuning_test_spearmanr_rval_closedloop_treadmill"] > 0.1)
    & (neurons_df["depth_tuning_test_spearmanr_pval_closedloop_treadmill"] < 0.05)
]
from scipy.stats import wilcoxon, mannwhitneyu
layer_color = {"Layer 2/3": "steelblue", "Layer 5": "crimson"}
if False:
    valid = ((neurons_df.rsof_test_rsq_closedloop_gratio_treadmill_trial_average_sig)
            | (neurons_df.rsof_test_rsq_closedloop_g2mult_treadmill_trial_average_sig)
            | (neurons_df.rsof_test_rsq_closedloop_g2d_treadmill_trial_average_sig))
else:
    valid = np.ones(len(neurons_df_sig), dtype=bool)
avg_by_sess = neurons_df_sig[valid].groupby(['session', 'layer']).mean(numeric_only=True).reset_index()
fig, ax = plt.subplots(1,1, figsize=(3,5))
plot_models = [f"rsof_test_rsq_closedloop_{mod}" for mod in 
               ['gratio_treadmill_trial_average', 
                'g2mult_treadmill_trial_average', 
                'g2d_treadmill_trial_average']
                ]
plot_labels = ['Depth', 'Depth x RS', 'Conjunctive']
between_layer_gap = 0.2
for isess, series in avg_by_sess.iterrows():
    l = series.layer == 'Layer 2/3'
    lcol = layer_color['Layer 2/3'] if l else layer_color['Layer 5']
    ax.plot(
        np.arange(len(plot_models)) + float(l) * between_layer_gap, 
        cl(series[plot_models]),
        mfc=lcol,
        mec=lcol,
        alpha=0.3,
        color=lcol,
        lw=0.5,
        marker='o')
# Add mean
avg_by_sess.groupby('layer')[plot_models].mean(numeric_only=True).T
for layer, lcol in layer_color.items():
    means = avg_by_sess[avg_by_sess.layer == layer][plot_models].mean(numeric_only=True).values
    ax.scatter(
        np.arange(len(plot_models)) + float(layer == 'Layer 2/3') * between_layer_gap, 
        cl(means),
        alpha=1.0,
        color=lcol,
        s=400,
        linewidths=6,
        marker='_',
        zorder=20)

ax.set_xticks(between_layer_gap/2+ np.arange(len(plot_models)), labels=plot_labels, rotation=90)
ax.set_xlim(-0.2, len(plot_models)-1 + between_layer_gap + 0.2)
ax.set_ylabel('Model $r^2$')
sns.despine()

PLOT_STARS = True
if PLOT_STARS:
    # --- Within-layer paired stats: Depth vs Depth x RS, Depth vs Conjunctive ---
    # Each session contributes one value per model -> paired across sessions.
    # Wilcoxon signed-rank (paired, non-parametric); two-sided.
    # Bonferroni correction applied across all tests below.
    def pstar(p):
        if p < 1e-3: return '***'
        if p < 1e-2: return '**'
        if p < 5e-2: return '*'
        return 'n.s.'

    comparisons = [(0, 1), (0, 2), (1, 2)]  # Depth vs Depth x RS, Depth vs Conjunctive, Depth x RS vs Conjunctive
    layer_offset = {'Layer 5': 0.0, 'Layer 2/3': 0.15}


    # 1) Run all tests, collect raw p-values
    #    within-layer: paired Wilcoxon (Depth vs Depth x RS, etc.)
    #    between-layer: is the within-session increase bigger in one layer than the other?
    #                   -> Mann-Whitney U on the per-session differences (independent samples)
    tests = []
    for layer in ['Layer 5', 'Layer 2/3']:
        sub = avg_by_sess[avg_by_sess.layer == layer]
        for i, j in comparisons:
            x = cl(sub[plot_models[i]]).values
            y = cl(sub[plot_models[j]]).values
            stat, p = wilcoxon(x, y)
            tests.append(dict(layer=layer, i=i, j=j, n=len(x),
                            med=float(np.median(x - y)), p=p))

    between_tests = []
    for i, j in comparisons:
        diffs = {}
        for layer in ['Layer 5', 'Layer 2/3']:
            sub = avg_by_sess[avg_by_sess.layer == layer]
            # increase from model i to model j, within session
            diffs[layer] = cl(sub[plot_models[j]]).values - cl(sub[plot_models[i]]).values
        stat, p = mannwhitneyu(diffs['Layer 2/3'], diffs['Layer 5'])
        between_tests.append(dict(i=i, j=j,
                                n23=len(diffs['Layer 2/3']), n5=len(diffs['Layer 5']),
                                med23=float(np.median(diffs['Layer 2/3'])),
                                med5=float(np.median(diffs['Layer 5'])), p=p))

    # 2) Bonferroni correction across ALL tests (within + between)
    n_tests = len(tests) + len(between_tests)
    for t in tests + between_tests:
        t['p_adj'] = min(t['p'] * n_tests, 1.0)

    # 3) Plot brackets (stars from corrected p) and print summary
    ydata_max = float(np.nanmax([cl(avg_by_sess[m]).max() for m in plot_models]))
    ymin = ax.get_ylim()[0]
    step = 0.07 * (ydata_max - ymin)
    level = 0
    print(f'Within-layer paired Wilcoxon signed-rank tests (two-sided, Bonferroni x{n_tests}):')
    for t in tests:
        off, col = layer_offset[t['layer']], layer_color[t['layer']]
        print(f"  {t['layer']:9s}  {plot_labels[t['i']]} vs {plot_labels[t['j']]:11s}  "
            f"n={t['n']}  median(diff)={t['med']:+.4f}  "
            f"p={t['p']:.4f}  p_Bonf={t['p_adj']:.4f}  {pstar(t['p_adj'])}")
        yb = ydata_max + step * (level + 1)
        x0, x1 = t['i'] + off, t['j'] + off
        ax.plot([x0, x0, x1, x1], [yb, yb + step*0.3, yb + step*0.3, yb],
                lw=1, color=col, clip_on=False)
        ax.text((x0 + x1) / 2, yb + step*0.3, pstar(t['p_adj']),
                ha='center', va='bottom', color=col, fontsize=11)
        level += 1

    # Between-layer brackets (does the increase differ by layer?), drawn in black above.
    print(f'\nBetween-layer Mann-Whitney U on per-session increases (two-sided, Bonferroni x{n_tests}):')
    mid = (layer_offset['Layer 5'] + layer_offset['Layer 2/3']) / 2
    for t in between_tests:
        print(f"  {plot_labels[t['i']]} -> {plot_labels[t['j']]:11s}  "
            f"L2/3 median(incr)={t['med23']:+.4f} (n={t['n23']})  "
            f"L5 median(incr)={t['med5']:+.4f} (n={t['n5']})  "
            f"p={t['p']:.4f}  p_Bonf={t['p_adj']:.4f}  {pstar(t['p_adj'])}")
        yb = ydata_max + step * (level + 1)
        x0, x1 = t['i'] + mid, t['j'] + mid
        ax.plot([x0, x0, x1, x1], [yb, yb + step*0.3, yb + step*0.3, yb],
                lw=1, color='k', clip_on=False)
        ax.text((x0 + x1) / 2, yb + step*0.3, pstar(t['p_adj']),
                ha='center', va='bottom', color='k', fontsize=11)
        level += 1
fig.savefig("rsq_d_dxrs_conj_by_layer.pdf", bbox_inches="tight")

In [ ]:
# Box plot of the difference in RSQ between the models, grouped by layer
# Averaged per session first (one point = one session/layer), with significance stars.
from scipy.stats import mannwhitneyu

def pstar(p):
    """Significance stars from a p-value."""
    if p < 1e-4: return '****'
    if p < 1e-3: return '***'
    if p < 1e-2: return '**'
    if p < 0.05: return '*'
    return 'n.s.'

# Model r^2 columns (trial-average, closed loop)
gratio_col = 'rsof_test_rsq_closedloop_gratio_treadmill_trial_average'   # Depth
g2mult_col = 'rsof_test_rsq_closedloop_g2mult_treadmill_trial_average'   # Depth x RS
g2d_col    = 'rsof_test_rsq_closedloop_g2d_treadmill_trial_average'      # Conjunctive

# Pairwise differences to show (label, minuend, subtrahend)
comparisons = [
    ('Depth x RS - Depth',       g2mult_col, gratio_col),
    ('Conjunctive - Depth x RS', g2d_col,    g2mult_col),
    ('Conjunctive - Depth',      g2d_col,    gratio_col),
]

layer_color = {"Layer 2/3": "steelblue", "Layer 5": "crimson"}

# Per-session, per-layer means of each model r^2
avg_by_sess = (
    neurons_df_sig
    .groupby(['session', 'layer'])
    .mean(numeric_only=True)
    .reset_index()
)

# Build long-form dataframe of per-session differences
records = []
for label, minuend, subtrahend in comparisons:
    diff = avg_by_sess[minuend].values - avg_by_sess[subtrahend].values
    for d, layer in zip(diff, avg_by_sess['layer'].values):
        if np.isfinite(d):
            records.append({'comparison': label, 'diff': d, 'layer': layer})
diff_df = pd.DataFrame(records)

fig, ax = plt.subplots(1, 1, figsize=(9, 5))
order = [c[0] for c in comparisons]

sns.boxplot(
    data=diff_df, x='comparison', y='diff', hue='layer',
    order=order, hue_order=['Layer 2/3', 'Layer 5'],
    palette=layer_color, showfliers=False, width=0.6, ax=ax,
)
# Overlay the individual sessions
sns.stripplot(
    data=diff_df, x='comparison', y='diff', hue='layer',
    order=order, hue_order=['Layer 2/3', 'Layer 5'],
    palette=layer_color, dodge=True, size=5, alpha=0.6,
    edgecolor='white', linewidth=0.5, ax=ax, legend=False,
)

ax.axhline(0, color='grey', ls='--', lw=1, zorder=-1)
ax.set_xlabel('')
ax.set_ylabel(r'$\Delta$ $R^2$', fontsize=18)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0, fontsize=18)
ax.set_yticks([0,0.1,0.2])
ax.set_xticklabels(ax.get_xticklabels(), fontsize=18)
ax.set_ylim(-0.05, 0.2)
ax.legend(title='', loc='upper right')
sns.despine(ax=ax)

# Between-layer significance stars, one per comparison group
y_hi = diff_df['diff'].max()
y_span = diff_df['diff'].max() - diff_df['diff'].min()
for i, (label, *_ ) in enumerate(comparisons):
    l23 = diff_df[(diff_df.comparison == label) & (diff_df.layer == 'Layer 2/3')]['diff']
    l5  = diff_df[(diff_df.comparison == label) & (diff_df.layer == 'Layer 5')]['diff']
    stat, p = mannwhitneyu(l23.values, l5.values, alternative='two-sided')

    # bracket spanning the two hue-dodged boxes
    x_l, x_r = i - 0.2, i + 0.2
    y_bar = y_hi + 0.06 * y_span
    y_txt = y_bar + 0.01 * y_span
    ax.plot([x_l, x_l, x_r, x_r],
            [y_bar, y_bar + 0.015 * y_span, y_bar + 0.015 * y_span, y_bar],
            lw=1.2, color='k')
    ax.text(i, y_txt + 0.015 * y_span, pstar(p),
            ha='center', va='bottom', fontsize=13)

    
plt.tight_layout()

# Print the underlying stats
for label, *_ in comparisons:
    l23 = diff_df[(diff_df.comparison == label) & (diff_df.layer == 'Layer 2/3')]['diff']
    l5  = diff_df[(diff_df.comparison == label) & (diff_df.layer == 'Layer 5')]['diff']
    stat, p = mannwhitneyu(l23.values, l5.values, alternative='two-sided')
    print(f"{label:26s}  L2/3 median={l23.median():+.3f} (n={len(l23)})  "
          f"L5 median={l5.median():+.3f} (n={len(l5)})  MWU p={p:.3g}  {pstar(p)}")

fig.savefig("rsq_differences_by_layer.pdf", bbox_inches="tight")


In [ ]:
# Box plot of the difference in RSQ between the models, grouped by layer
# Averaged per session first (one point = one session/layer), with significance stars.
from scipy.stats import mannwhitneyu

def pstar(p):
    """Significance stars from a p-value."""
    if p < 1e-4: return '****'
    if p < 1e-3: return '***'
    if p < 1e-2: return '**'
    if p < 0.05: return '*'
    return 'n.s.'

# Model r^2 columns (trial-average, closed loop)
gratio_col = 'rsof_test_rsq_closedloop_gratio_treadmill_trial_average'   # Depth
g2mult_col = 'rsof_test_rsq_closedloop_g2mult_treadmill_trial_average'   # Depth x RS
g2d_col    = 'rsof_test_rsq_closedloop_g2d_treadmill_trial_average'      # Conjunctive

# Pairwise differences to show (label, minuend, subtrahend)
comparisons = [
    ('Depth x RS - Depth',       g2mult_col, gratio_col),
    ('Conjunctive - Depth x RS', g2d_col,    g2mult_col),
]

layer_color = {"Layer 2/3": "steelblue", "Layer 5": "crimson"}

# Per-session, per-layer means of each model r^2
avg_by_sess = (
    neurons_df_sig
    .groupby(['session', 'layer'])
    .mean(numeric_only=True)
    .reset_index()
)

# Build long-form dataframe of per-session differences
records = []
for label, minuend, subtrahend in comparisons:
    diff = avg_by_sess[minuend].values - avg_by_sess[subtrahend].values
    for d, layer in zip(diff, avg_by_sess['layer'].values):
        if np.isfinite(d):
            records.append({'comparison': label, 'diff': d, 'layer': layer})
diff_df = pd.DataFrame(records)

fig, ax = plt.subplots(1, 1, figsize=(7, 3))
order = [c[0] for c in comparisons]

sns.boxplot(
    data=diff_df, y='comparison', x='diff', hue='layer',
    order=order, hue_order=['Layer 2/3', 'Layer 5'],
    palette=layer_color, showfliers=False, width=0.6, ax=ax,
)
# Overlay the individual sessions
sns.stripplot(
    data=diff_df, y='comparison', x='diff', hue='layer',
    order=order, hue_order=['Layer 2/3', 'Layer 5'],
    palette=layer_color, dodge=True, size=10, alpha=0.6,
    edgecolor='white', linewidth=0.5, ax=ax, legend=False,
)

ax.axvline(0, color='grey', ls='--', lw=1, zorder=-1)
ax.set_ylabel('')
ax.set_xlabel(r'$\Delta$ $R^2$', fontsize=18)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=18)
ax.set_xticks([0, 0.1, 0.2])
ax.set_xticklabels(ax.get_xticklabels(), fontsize=18)
ax.set_xlim(-0.05, 0.2)
l = ax.legend(title='', loc='lower right')
sns.despine(ax=ax)
l.remove()
# Between-layer significance stars, one per comparison group
x_hi = diff_df['diff'].max()
x_span = diff_df['diff'].max() - diff_df['diff'].min()
for i, (label, *_ ) in enumerate(comparisons):
    l23 = diff_df[(diff_df.comparison == label) & (diff_df.layer == 'Layer 2/3')]['diff']
    l5  = diff_df[(diff_df.comparison == label) & (diff_df.layer == 'Layer 5')]['diff']
    stat, p = mannwhitneyu(l23.values, l5.values, alternative='two-sided')

    # bracket spanning the two hue-dodged boxes (now vertical)
    y_l, y_r = i - 0.2, i + 0.2
    x_bar = x_hi + 0.06 * x_span
    x_txt = x_bar + 0.01 * x_span
    ax.plot([x_bar, x_bar + 0.015 * x_span, x_bar + 0.015 * x_span, x_bar],
            [y_l, y_l, y_r, y_r],
            lw=1.2, color='k')
    ax.text(x_txt + 0.015 * x_span, i, pstar(p),
            ha='left', va='center', fontsize=13)

plt.tight_layout()

# Print the underlying stats
for label, *_ in comparisons:
    l23 = diff_df[(diff_df.comparison == label) & (diff_df.layer == 'Layer 2/3')]['diff']
    l5  = diff_df[(diff_df.comparison == label) & (diff_df.layer == 'Layer 5')]['diff']
    stat, p = mannwhitneyu(l23.values, l5.values, alternative='two-sided')
    print(f"{label:26s}  L2/3 median={l23.median():+.3f} (n={len(l23)})  "
          f"L5 median={l5.median():+.3f} (n={len(l5)})  MWU p={p:.3g}  {pstar(p)}")

fig.savefig("rsq_differences_by_layer.pdf", bbox_inches="tight")


In [ ]:
# --- Does the relative Depth -> Depth x RS gain differ between layers? ---
# Most sessions have only one layer, so compare layers unpaired (Mann-Whitney U).
# Relative gain controls for layers having different baseline r^2.
from scipy.stats import mannwhitneyu
plot_models = [f"rsof_test_rsq_closedloop_{mod}" for mod in 
               ['gratio_treadmill_trial_average', 
                'g2mult_treadmill_trial_average', 
                'g2d_treadmill_trial_average']
                ]

depth_col   = plot_models[0]  # 'Depth'
depthrs_col = plot_models[1]  # 'Depth x RS'

d = avg_by_sess.copy()
d['rel_gain'] = (d[depthrs_col].values - d[depth_col].values) / d[depth_col].values
d['abs_gain'] = (d[depthrs_col].values - d[depth_col].values)

l23 = d[d.layer == 'Layer 2/3']['rel_gain']
l5  = d[d.layer == 'Layer 5']['rel_gain']
stat, p = mannwhitneyu(l23.values, l5.values, alternative='two-sided')
print('Relative gain (Depth x RS - Depth) / Depth, per session:')
print(f"  L2/3: n={len(l23)}  median={l23.median():+.3%}")
print(f"  L5:   n={len(l5)}  median={l5.median():+.3%}")
print(f"  Mann-Whitney U (two-sided): U={stat:.0f}, p={p:.4f}  {pstar(p)}")


l23 = d[d.layer == 'Layer 2/3']['abs_gain']
l5  = d[d.layer == 'Layer 5']['abs_gain']
stat, p = mannwhitneyu(l23.values, l5.values, alternative='two-sided')
print('Absolute gain (Depth x RS - Depth), per session:')
print(f"  L2/3: n={len(l23)}  median={l23.median():.2f}")
print(f"  L5:   n={len(l5)}  median={l5.median():.2f}")
print(f"  Mann-Whitney U (two-sided): U={stat:.0f}, p={p:.4f}  {pstar(p)}")


In [ ]:
fig =plt.figure(figsize=(15,5))

ax0 = fig.add_subplot(1,2,1)
ax1 = fig.add_subplot(1,2,2)
temp_df = []
for iax, layer in enumerate(['Layer 2/3', 'Layer 5']):
    # Calculate percentage of neurons that have the best model
    subset_df = neurons_df_sig[neurons_df_sig.layer == layer]
    g2d= subset_df['rsof_test_rsq_closedloop_g2d_treadmill_trial_average'].values
    gratio = subset_df['rsof_test_rsq_closedloop_g2mult_treadmill_trial_average'].values
    
    df = pd.DataFrame(dict(g2d=g2d, gratio=gratio, layer=layer, session=subset_df.session.values, diff_conjratio=g2d-gratio))
    bad = pd.isna(df.g2d) | pd.isna(df.gratio)
    temp_df.append(df[(~bad)])
temp_df = pd.concat(temp_df, ignore_index=True) 
temp_df['diff_conjratio'] = np.clip(temp_df['diff_conjratio'], -1, 1)
_ = sns.histplot(temp_df, x='diff_conjratio', hue='layer',stat='density', common_norm=False, binrange=(-0.05,0.2),  binwidth=0.002, element="step", cumulative=False, ax=ax0, kde=False)
_ = sns.histplot(temp_df, x='diff_conjratio', hue='layer',stat='density', common_norm=False, binrange=(-0.05,0.2), binwidth=1e-3, element="step", cumulative=True, ax=ax1)
ax1.axhline(1, color='grey', ls='--', zorder=-2)


for ax, h in zip([ax0, ax1], [100, 1.05]):
    for l, d in temp_df.groupby('layer'):
        ax.scatter(np.nanmean(d.diff_conjratio), h, marker='v', color='C0' if l == 'Layer 2/3' else 'C1', label=f'Mean {l}')
        ax.scatter(np.nanmedian(d.diff_conjratio), h, marker='|', color='C0' if l == 'Layer 2/3' else 'C1', label=f'Median {l}')

    ax.set_xlabel('Conjunctive $R^2$ - Depth x RS $R^2$')
ax1.legend(loc='lower right')
l = ax0.legend()
l.set_visible(False)
plt.tight_layout()

In [ ]:
fig =plt.figure(figsize=(15,5))

ax0 = fig.add_subplot(1,2,1)
ax1 = fig.add_subplot(1,2,2)
temp_df = []
for layer in ['Layer 2/3', 'Layer 5']:
    # Calculate percentage of neurons that have the best model
    subset_df = neurons_df_sig[neurons_df_sig.layer == layer]
    g2mult= subset_df['rsof_test_rsq_closedloop_g2mult_treadmill_trial_average'].values
    gratio = subset_df['rsof_test_rsq_closedloop_gratio_treadmill_trial_average'].values
    g2d = subset_df['rsof_test_rsq_closedloop_g2d_treadmill_trial_average'].values
    df = pd.DataFrame(dict(g2d=g2mult, gratio=gratio, layer=layer, session=subset_df.session.values, diff_conjratio=g2mult-gratio))
    bad = pd.isna(df.g2d) | pd.isna(df.gratio)
    temp_df.append(df[~bad])
temp_df = pd.concat(temp_df, ignore_index=True) 
temp_df['diff_conjratio'] = np.clip(temp_df['diff_conjratio'], -1, 1)
_ = sns.histplot(temp_df, x='diff_conjratio', hue='layer',stat='density', common_norm=False, binrange=(-0.05,0.2),  binwidth=0.002, element="step", cumulative=False, ax=ax0, kde=False)
_ = sns.histplot(temp_df, x='diff_conjratio', hue='layer',stat='density', common_norm=False, binrange=(-0.05,0.2), binwidth=1e-3, element="step", cumulative=True, ax=ax1)
ax1.axhline(1, color='grey', ls='--', zorder=-2)


for ax, h in zip([ax0, ax1], [100, 1.05]):
    for l, d in temp_df.groupby('layer'):
        ax.scatter(np.nanmean(d.diff_conjratio), h, marker='v', color='C0' if l == 'Layer 2/3' else 'C1', label=f'Mean {l}')
        ax.scatter(np.nanmedian(d.diff_conjratio), h, marker='|', color='C0' if l == 'Layer 2/3' else 'C1', label=f'Median {l}')

    ax.set_xlabel('Depth x RS $R^2$ - Depth  $R^2$')
ax1.legend(loc='lower right')
l = ax0.legend()
l.set_visible(False)
plt.tight_layout()

In [ ]:
fig =plt.figure(figsize=(10,5))


for iax, layer in enumerate(['Layer 2/3', 'Layer 5']):

    ax = fig.add_subplot(1,2,1 + iax)
    markersize=10
    alpha=0.7
    color='k'
    fontsize_dict={"title": 15, "label": 10, "tick": 10}
    ci=None


    # Calculate percentage of neurons that have the best model
    subset_df = neurons_df_sig[neurons_df_sig.layer == layer]
    neuron_sum = (
        subset_df.groupby("session")[["roi"]].agg(["count"]).values.flatten()
    )
    props = []
    # calculate the proportion of neurons that have the best model for each session
    for i, model in enumerate(model_cols):
        prop = (
            subset_df.groupby("session")[["best_model", "roi"]]
            .apply(lambda x: x[x["best_model"] == model][["roi"]].agg(["count"]))
            .values.flatten()
        ) / neuron_sum
        props.append(prop)
        # Plot bar plot
    for i, model in enumerate(model_cols):
        sns.stripplot(
            x=np.ones(len(props[i])) * i,
            y=props[i],
            size=markersize,
            alpha=alpha,
            jitter=0.4,
            edgecolor="white",
            color=sns.color_palette("dark")[i],
        )
        plt.plot(
            [i - 0.4, i + 0.4],
            [np.median(props[i]), np.median(props[i])],
            linewidth=3,
            color=color,
        )
        if ci is not None:
            plt.fill_between(
                [i - 0.4, i + 0.4],
                [ci[i][0], ci[i][0]],
                [ci[i][1], ci[i][1]],
                color=sns.color_palette("dark")[i],
                alpha=0.7,
                edgecolor="none",
            )

    sns.despine(offset=5, ax=plt.gca())
    ax.set_xticks(range(len(models)))
    ax.set_xticklabels(model_labels, fontsize=fontsize_dict["label"], rotation=90)
    ax.tick_params(axis='y', labelsize=fontsize_dict["label"])
    ax.set_ylabel(
        "Proportion of neurons with best model fit",
        fontsize=fontsize_dict["label"],
    )
    ax.set_ylim([0, 0.5])
    ax.set_title(layer)


In [ ]:
neurons_df["preferred_depth_amplitude"] = neurons_df[
    "depth_tuning_popt_closedloop_treadmill"
].apply(lambda x: np.exp(x[0]) + x[-1])
neurons_df_sig = neurons_df[
    (neurons_df["depth_tuning_test_spearmanr_rval_closedloop_treadmill"] > 0.1)
    & (neurons_df["depth_tuning_test_spearmanr_pval_closedloop_treadmill"] < 0.05)
]

In [ ]:
popt_col = 'rsof_popt_closedloop_g2d_treadmill_trial_average'
min_sigma = 0.25

valid = neurons_df_sig[popt_col].apply(lambda x: isinstance(x, np.ndarray) and len(x) >= 5)
df = neurons_df_sig[valid].copy()

popt_arr = np.vstack(df[popt_col].values)
# sigma = sqrt(exp(log_sigma2) + min_sigma), in log-space units (log m/s, log °/s)
df['sigma_rs'] = np.sqrt(np.exp(popt_arr[:, 3]) + min_sigma)
df['sigma_of'] = np.sqrt(np.exp(popt_arr[:, 4]) + min_sigma)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col, label in zip(
    axes,
    ['sigma_rs', 'sigma_of'],
    [r'RS $\sigma$ (log m/s)', r'OF $\sigma$ (log °/s)'],
):
    sns.histplot(
        df, x=col, hue='layer', stat='density', common_norm=False,
        binwidth=0.1, element='step', ax=ax, binrange=(0, 5),
    )
    for layer, color in zip(['Layer 2/3', 'Layer 5'], ['C0', 'C1']):
        med = np.nanmedian(df.loc[df.layer == layer, col])
        ax.axvline(med, color=color, ls='--', lw=1.5, label=f'Median {layer}: {med:.2f}')
    ax.set_xlabel(label)
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)
sns.despine()
plt.tight_layout()


In [ ]:
neurons_df["preferred_depth_amplitude"] = neurons_df[
    "depth_tuning_popt_closedloop_treadmill"
].apply(lambda x: np.exp(x[0]) + x[-1])
neurons_df_sig = neurons_df[
    (neurons_df["depth_tuning_test_spearmanr_rval_closedloop_treadmill"] > 0.1)
    & (neurons_df["depth_tuning_test_spearmanr_pval_closedloop_treadmill"] < 0.05)
    #& (neurons_df["rsof_rsq_closedloop_g2mult_treadmill_trial_average"] > 0.3)
]

In [ ]:
popt_col = 'rsof_popt_closedloop_gprod_treadmill_trial_average'


neurons_df["preferred_depth_amplitude"] = neurons_df[
    "depth_tuning_popt_closedloop_treadmill"
].apply(lambda x: np.exp(x[0]) + x[-1])
neurons_df_sig = neurons_df[
    (neurons_df["depth_tuning_test_spearmanr_rval_closedloop_treadmill"] > 0.1)
    & (neurons_df["depth_tuning_test_spearmanr_pval_closedloop_treadmill"] < 0.05)
    #& (neurons_df["rsof_rsq_closedloop_g2mult_treadmill_trial_average"] > 0.3)
]

min_sigma = 0.25

valid = neurons_df_sig[popt_col].apply(lambda x: isinstance(x, np.ndarray) and len(x) >= 5)
df = neurons_df_sig[valid].copy()

popt_arr = np.vstack(df[popt_col].values)
# gprod: index 3 = log_sigma_x2 (RS), index 4 = log_sigma_y2 (OF)
df['sigma_of'] = np.clip((np.sqrt(np.exp(popt_arr[:, 4]) + min_sigma))/np.log(2), 0, 7)
df['sigma_rs'] = np.clip((np.sqrt(np.exp(popt_arr[:, 3]) + min_sigma))/np.log(2), 0, 7)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
palette = {'Layer 2/3': 'C0', 'Layer 5': 'C1'}

fold_ticks = [1, 2, 4, 8, 16, 32]

for ax, col, label in zip(
    axes,
    ['sigma_of', 'sigma_rs'],
    [r'OF tuning σ (octaves)', r'RS tuning σ (octaves)'],
):
    sns.histplot(df, x=col, hue='layer', palette=palette, stat='density', common_norm=False,
                 element='step', ax=ax, binwidth=0.2, binrange=(0, 7))
    ax.set_xticks([np.log2(k) for k in fold_ticks])
    ax.set_xticklabels([f'{k}' for k in fold_ticks])
    ax.set_xlabel(label)
    for layer, color in zip(['Layer 2/3', 'Layer 5'], ['C0', 'C1']):
        med = np.nanmedian(df.loc[df.layer == layer, col])
        ax.scatter(med, 1, color=color, marker='v', s=50, label=f'Median {layer}: {med:.1f}×')
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)
sns.despine()
plt.tight_layout()


In [ ]:
len(neurons_df_sig)/len(neurons_df)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)

neurons_df["preferred_depth_amplitude"] = neurons_df[
    "depth_tuning_popt_closedloop_treadmill"
].apply(lambda x: np.exp(x[0]) + x[-1])
neurons_df_sig = neurons_df[
    (neurons_df["depth_tuning_test_spearmanr_rval_closedloop_treadmill"] > 0)
    & (neurons_df["depth_tuning_test_spearmanr_pval_closedloop_treadmill"] < 0.05)
    #& (neurons_df["rsof_rsq_closedloop_g2mult_treadmill_trial_average"] > 0.3)
]

rs_col    = 'rsof_test_rsq_closedloop_grs_treadmill_trial_average'
of_col    = 'rsof_test_rsq_closedloop_gof_treadmill_trial_average'
depth_col = 'rsof_test_rsq_closedloop_gratio_treadmill_trial_average'

vmin, vmax = 0, 0.6

for ax, layer in zip(axes, ['Layer 2/3', 'Layer 5']):
    sub = neurons_df_sig[neurons_df_sig.layer == layer]
    print(f'Layer: {layer}, n={len(sub)}')
    x = cl(sub[rs_col])
    y = cl(sub[of_col])
    c = np.clip(sub[depth_col], vmin, vmax)
    sc = ax.scatter(x, y, c=c, cmap='viridis', vmin=vmin, vmax=vmax,
                    alpha=0.3, s=10, edgecolors='none', rasterized=True)
    ax.plot([-0.1, 1], [-0.1, 1], color='grey', ls='--', lw=1, alpha=0.4, zorder=-1)
    ax.set_xlim(-0.1, 1)
    ax.set_ylim(-0.1, 1)
    ax.set_xlabel('RS model $r^2$')
    ax.set_ylabel('OF model $r^2$')
    ax.set_title(layer)
    ax.set_aspect('equal')

cbar = fig.colorbar(sc, ax=axes, shrink=0.8, label='Depth model $r^2$')
sns.despine()



In [ ]:
import itertools

cols = {
    'RS $r^2$':    rs_col,
    'OF $r^2$':    of_col,
    'Depth $r^2$': depth_col,
}

plot_df = pd.concat([
    neurons_df_sig[neurons_df_sig.layer == layer][list(cols.values()) + ['layer']]
    for layer in ['Layer 2/3', 'Layer 5']
    
]).copy()

for col in cols.values():
    plot_df[col] = cl(plot_df[col])

plot_df = plot_df.rename(columns={v: k for k, v in cols.items()})

g = sns.PairGrid(plot_df, hue='layer', palette={'Layer 2/3': 'C0', 'Layer 5': 'C1'},
                 vars=list(cols.keys()))
g.map_offdiag(plt.scatter, alpha=0.15, s=5, edgecolors='none', rasterized=True)
g.map_diag(sns.histplot, element='step', stat='density', common_norm=False)
g.add_legend()


In [ ]:
[c for c in neurons_df.columns if 'gadd_treadmill_trial_average' in c]

In [ ]:
[c for c in neurons_df.columns if '_sig' in c]

In [ ]:
popt_col = 'rsof_popt_closedloop_g2mult_treadmill_trial_average'
min_sigma = 0.25

neurons_df_sig = neurons_df[
    (neurons_df["depth_tuning_test_spearmanr_rval_closedloop_treadmill"] > 0.1)
    & (neurons_df["depth_tuning_test_spearmanr_pval_closedloop_treadmill"] < 0.05)
]

valid = neurons_df_sig[popt_col].apply(lambda x: isinstance(x, np.ndarray) and len(x) >= 5)
df = neurons_df_sig[valid].copy()
# Keep only cells whose best model is gratio or g2mult AND at least one passes significance
best_model_cols = {
    'rsof_test_rsq_closedloop_gratio_treadmill_trial_average',
    'rsof_test_rsq_closedloop_g2mult_treadmill_trial_average',
}
gratio_sig = neurons_df.loc[df.index, 'rsof_test_rsq_closedloop_gratio_treadmill_trial_average_sig']
g2mult_sig = neurons_df.loc[df.index, 'rsof_test_rsq_closedloop_g2mult_treadmill_trial_average_sig']
is_best    = neurons_df.loc[df.index, 'best_model'].isin(best_model_cols)
df = df[is_best & (gratio_sig | g2mult_sig)]

popt_arr = np.vstack(df[popt_col].values)
# Store in log2 (octaves) — log2(exp(sigma)) = sigma/log(2), same thing
df['sigma_rsof_ratio'] = np.clip((np.sqrt(np.exp(popt_arr[:, 3]) + min_sigma))/np.log(2), 0, 7)
df['sigma_rs']         = np.clip((np.sqrt(np.exp(popt_arr[:, 4]) + min_sigma))/np.log(2), 0, 7)

print(f"n={len(df)} neurons where best model is gratio or g2mult and R² is significant")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
palette = {'Layer 2/3': 'C0', 'Layer 5': 'C1'}

fold_ticks = [1, 2, 4, 8, 16, 32]

for ax, col, label in zip(
    axes,
    ['sigma_rsof_ratio', 'sigma_rs'],
    [r'Depth tuning σ (octaves)', r'RS tuning σ (octaves)'],
):
    sns.histplot(df, x=col, hue='layer', palette=palette, stat='density', common_norm=False,
                 element='step', ax=ax, binwidth=0.2, binrange=(0, 7))
    ax.set_xticks([np.log2(k) for k in fold_ticks])
    ax.set_xticklabels([f'{k}' for k in fold_ticks])
    ax.set_xlabel(label)
    for layer, color in zip(['Layer 2/3', 'Layer 5'], ['C0', 'C1']):
        med = np.nanmedian(df.loc[df.layer == layer, col])
        ax.scatter(med, 1, color=color, marker='v', s=50, label=f'Median {layer}: {med:.1f}×')
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)
sns.despine()
plt.tight_layout()


In [ ]:
rs_pval_col = 'rsof_test_spearmanr_pval_closedloop_grs_treadmill_trial_average'
of_pval_col = 'rsof_test_spearmanr_pval_closedloop_gof_treadmill_trial_average'

neurons_df['rs_spearman_sig'] = neurons_df[rs_pval_col] < 0.05
neurons_df['of_spearman_sig'] = neurons_df[of_pval_col] < 0.05

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
palette = {'Layer 2/3': 'C0', 'Layer 5': 'C1'}
categories = ['R²', 'Spearman', 'Both', 'Neither']

for ax, rsq_sig_col, spearman_sig_col, label in zip(
    axes,
    ['rs_sig', 'of_sig'],
    ['rs_spearman_sig', 'of_spearman_sig'],
    ['RS', 'OF'],
):
    records = []
    for layer in ['Layer 2/3', 'Layer 5']:
        mask = neurons_df['layer'] == layer
        n = mask.sum()
        rsq = neurons_df.loc[mask, rsq_sig_col]
        spe = neurons_df.loc[mask, spearman_sig_col]
        records.append({'layer': layer, 'criterion': 'R²',      'frac': rsq.sum() / n})
        records.append({'layer': layer, 'criterion': 'Spearman', 'frac':  spe.sum() / n})
        records.append({'layer': layer, 'criterion': 'Both',     'frac': ((rsq) & (spe)).sum() / n})
        records.append({'layer': layer, 'criterion': 'Neither',    'frac': ((~rsq) & (~spe)).sum() / n})

    plot_df = pd.DataFrame(records)
    sns.barplot(data=plot_df, x='criterion', y='frac', hue='layer',
                palette=palette, order=categories, ax=ax)
    ax.set_title(label)
    ax.set_xlabel('')
    ax.set_ylabel('Fraction of neurons')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha='right')
    ax.legend(title='')
    sns.despine(ax=ax)

plt.tight_layout()

In [ ]:
plot_df

In [ ]:
# Example cells illustrating the RS/OF fits: one representative cell per layer.
# Columns: measured RS/OF matrix | Depth (gratio) | Depth x RS (g2mult) | Conjunctive (g2d)
import scipy.stats
from matplotlib.colors import ListedColormap
import cottage_analysis.analysis.fit_gaussian_blob as fit_gb
from cottage_analysis.pipelines import pipeline_utils

SFX = "_treadmill_trial_average"
MIN_SIGMA = 0.25
RS_THR = 0.01
MAX_RS2MOTOR_DIFF = 0.3
LOG_RANGE = {"rs_bin_log_min": 0, "rs_bin_log_max": 2.5,
             "of_bin_log_min": -1.5, "of_bin_log_max": 3.5}
fit_models = [("gratio", "Depth"), ("g2mult", "Depth x RS"), ("g2d", "Conjunctive")]
sess_to_project = neurons_df.groupby("session")["project"].first().to_dict()

# Nominal treadmill stimulus grid (from decoder_treadmill_cut.ipynb), in display units.
RS_GRID = np.log10(np.array([4, 8.0, 16.0, 32.0, 64.0]))          # log10 cm/s
OF_GRID = np.log10(np.array([1.0, 4.0, 16.0, 64.0, 256.0, 1024.0]))    # log10 deg/s


def _grid_axis_limits(grid):
    """(lo, hi) in log units, half a bin beyond the outermost presented grid values."""
    return grid[0] - (grid[1] - grid[0]) / 2, grid[-1] + (grid[-1] - grid[-2]) / 2


RS_LIM = _grid_axis_limits(RS_GRID)   # bins centred on ~4-60 cm/s
OF_LIM = _grid_axis_limits(OF_GRID)   # bins centred on 1-1024 deg/s


def _valid_popt(x, n_min=4):
    x = np.asarray(x, dtype=float).ravel() if isinstance(x, (list, np.ndarray)) else np.array([])
    return x.size >= n_min and np.all(np.isfinite(x))


def _photodiode(session):
    return 2 if ("PZAH6.4b" in session or "PZAG3.4f" in session) else 5


def load_session_trials(project, session):
    _, _, _, trials_df = pipeline_utils.load_session(
        project=project, session_name=session,
        photodiode_protocol=_photodiode(session), regenerate_frames=False,
        filter_datasets=dict(annotated=True), protocol_base="SpheresTubeMotor")
    return trials_df[~trials_df.recording_name.str.contains("multidepth")]


def reconstruct_trials(trials_df, roi):
    """Per-trial averaged (rs_log, of_log_deg, dff) for one ROI (closed loop, running-masked)."""
    tdf = trials_df[trials_df.closed_loop == 1]
    rs_l, of_l, dff_l = [], [], []
    has_diff = "max_abs_rs2motor_diff_ratio_stim" in tdf.columns
    for _, tr in tdf.iterrows():
        rs = np.asarray(tr["RS_stim"], dtype=float)
        rse = np.asarray(tr["RS_eye_stim"], dtype=float)
        of = np.asarray(tr["OF_stim"], dtype=float)
        dff = np.asarray(tr["dff_stim"], dtype=float)[:, roi]
        run = (rs > RS_THR) & (rse > RS_THR) & (~np.isnan(of)) & (of > 0)
        if has_diff:
            run = run & (np.asarray(tr["max_abs_rs2motor_diff_ratio_stim"], dtype=float) < MAX_RS2MOTOR_DIFF)
        if run.sum() == 0:
            continue
        rs_l.append(np.mean(rs[run]))
        of_l.append(np.mean(of[run]))
        dff_l.append(np.mean(dff[run]))
    rs_l, of_l, dff_l = np.array(rs_l), np.array(of_l), np.array(dff_l)
    return np.log(rs_l), np.log(np.degrees(of_l)), dff_l


def _present_edges(vals, grid):
    """Bin edges (log units) centred on the grid values that actually occur in `vals`."""
    present = np.sort(np.unique(grid[np.abs(vals[:, None] - grid[None, :]).argmin(1)]))
    if len(present) == 1:
        return np.array([present[0] - 0.25, present[0] + 0.25])
    mids = (present[:-1] + present[1:]) / 2
    return np.concatenate([[present[0] - (mids[0] - present[0])], mids,
                           [present[-1] + (present[-1] - mids[-1])]])


def plot_data_matrix(ax, trials_df, roi):
    """Measured matrix, binned on the actually-present RS/OF stimulus combinations. Returns vmax."""
    rs_log, of_log, dff = reconstruct_trials(trials_df, roi)
    x = np.log10(np.exp(rs_log) * 100)   # log10 cm/s
    y = np.log10(np.exp(of_log))         # log10 deg/s
    rs_edges, of_edges = _present_edges(x, RS_GRID), _present_edges(y, OF_GRID)
    M, _, _, _ = scipy.stats.binned_statistic_2d(x, y, dff, statistic="mean", bins=[rs_edges, of_edges])
    vmax = float(np.nanmax(M)) if np.isfinite(M).any() else 1.0
    cmap = plt.cm.Reds.copy(); cmap.set_bad("0.6")   # no-data bins -> solid grey
    ax.pcolormesh(rs_edges, of_edges, np.ma.masked_invalid(M.T), cmap=cmap, vmin=0, vmax=vmax)
    return vmax, rs_edges, of_edges, ~np.isfinite(M)


def render_fit(ax, session_df, roi, model, vmax):
    """Fitted RS/OF matrix (mirrors rsof_plots.plot_RS_OF_fit, + g2mult support)."""
    popt = np.asarray(session_df.loc[session_df.roi == roi,
                      f"rsof_popt_closedloop_{model}{SFX}"].iloc[0], dtype=float)
    lr = LOG_RANGE
    rs = np.logspace(lr["rs_bin_log_min"], lr["rs_bin_log_max"], 100, base=10) / 100  # m/s
    of = np.logspace(lr["of_bin_log_min"], lr["of_bin_log_max"], 100, base=10)         # deg/s
    rs_grid, of_grid = np.meshgrid(np.log(rs), np.log(of))
    funcs = {"g2d": fit_gb.gaussian_2d, "gratio": fit_gb.gaussian_1d,
             "g2mult": fit_gb.gaussian_multiplicative}
    params = (rs_grid - of_grid) if model == "gratio" else (rs_grid, of_grid)
    pred = funcs[model](params, *popt, min_sigma=MIN_SIGMA).reshape((len(of), len(rs)))
    extent = [lr["rs_bin_log_min"], lr["rs_bin_log_max"], lr["of_bin_log_min"], lr["of_bin_log_max"]]
    ax.imshow(pred, origin="lower", extent=extent, aspect="auto", cmap="Reds", vmin=0, vmax=vmax)


def overlay_nodata(ax, rs_edges, of_edges, nodata):
    """Overlay the measured matrix's no-data bins as a semi-transparent grey layer."""
    grey = np.where(nodata, 1.0, np.nan)
    ax.pcolormesh(rs_edges, of_edges, np.ma.masked_invalid(grey.T),
                  cmap=ListedColormap(["0.6"]), vmin=0, vmax=1, alpha=0.6, zorder=5)


def _sigma_rs_g2mult(x):
    """Intrinsic (unfloored) RS-axis sigma of the Depth x RS fit: sqrt(exp(log_sigma_y2))."""
    x = np.asarray(x, dtype=float).ravel() if isinstance(x, (list, np.ndarray)) else np.array([])
    if x.size < 6 or not np.all(np.isfinite(x)):
        return np.nan
    return np.sqrt(np.exp(x[4]))


def _style_axis(ax):
    # Restrict the view to the presented stimulus range (RS grid ~4-60 cm/s, OF grid 1-1024 deg/s).
    ax.set_xlim(*RS_LIM)
    ax.set_ylim(*OF_LIM)
    ax.set_aspect("equal")
    ax.set_xticks(RS_GRID); ax.set_xticklabels([f"{10 ** v:.0f}" for v in RS_GRID])
    ax.set_yticks(OF_GRID); ax.set_yticklabels([f"{10 ** v:.0f}" for v in OF_GRID])


# --- Select a representative example cell per layer ---
# Criteria (user-requested): depth-tuned, depth-tuning fit amplitude > 0.1 dff, and an
# RS sigma (Depth x RS / g2mult fit) in the top 75% of the pool (i.e. not in the bottom
# quartile of diagonal lengths). Among the survivors, take the cell whose Conjunctive
# (g2d) R2 is closest to the per-layer median, as a representative fit.
rsq_cols = {m: f"rsof_test_rsq_closedloop_{m}{SFX}" for m, _ in fit_models}
g2d_rsq = rsq_cols["g2d"]


def _depth_amplitude(popt):
    """Depth-tuning fit amplitude in dff: exp(log_amplitude) = exp(popt[0]) of gaussian_1d."""
    popt = np.asarray(popt, dtype=float).ravel() if isinstance(popt, (list, np.ndarray)) else np.array([])
    return np.exp(popt[0]) if popt.size >= 1 and np.isfinite(popt[0]) else np.nan


depth_tuned = (
    (neurons_df["depth_tuning_test_spearmanr_rval_closedloop_treadmill"] > 0.1)
    & (neurons_df["depth_tuning_test_spearmanr_pval_closedloop_treadmill"] < 0.05)
)
popt_ok = np.ones(len(neurons_df), dtype=bool)
rsq_ok = np.ones(len(neurons_df), dtype=bool)
for model, _ in fit_models:
    n_min = 7 if model == "g2d" else (4 if model == "gratio" else 6)
    popt_ok &= neurons_df[f"rsof_popt_closedloop_{model}{SFX}"].apply(lambda x: _valid_popt(x, n_min))
    rsq_ok &= neurons_df[rsq_cols[model]].between(-1, 1)

depth_amp = neurons_df["depth_tuning_popt_closedloop_treadmill"].apply(_depth_amplitude)
amp_ok = depth_amp > 0.1
sigma_rs = neurons_df[f"rsof_popt_closedloop_g2mult{SFX}"].apply(_sigma_rs_g2mult)

# Pool that passes the tuning / amplitude / validity gates; the RS-sigma cut is relative to it.
pool = depth_tuned & popt_ok & rsq_ok & amp_ok & np.isfinite(sigma_rs)
sigma_p25 = np.nanpercentile(sigma_rs[pool], 25)   # bottom of the "top 75%"
sigma_ok = sigma_rs >= sigma_p25
cand = neurons_df[pool & sigma_ok].copy()
cand["_sigma_rs"] = sigma_rs[pool & sigma_ok]
cand["_depth_amp"] = depth_amp[pool & sigma_ok]
print(f"Pool n={int(pool.sum())}; RS-sigma top-75% cutoff (p25) = {sigma_p25:.2f}; "
      f"candidates n={len(cand)}")

# Manual overrides: force a specific (session, roi) per layer; layers not listed here
# fall back to the automatic median-g2d-R2 pick from the candidate pool above.
# Layer 5:   PZAG22.1a_S20260206 roi 320 (nominal depth ~491 um)
# Layer 2/3: PZAG22.1a_S20260307b roi 80 (nominal depth ~300 um)  [alternatives: 0, 308]
FORCE = {
    "Layer 5": ("PZAG22.1a_S20260210", 143),
    "Layer 2/3": ("PZAG22.1a_S20260307b", 80),
}

examples = {}
for layer in ["Layer 5", "Layer 2/3"]:
    if layer in FORCE:
        session, roi = FORCE[layer]
        pick = neurons_df[(neurons_df.session == session) & (neurons_df.roi == roi)].iloc[0].copy()
        pick["_depth_amp"] = _depth_amplitude(pick["depth_tuning_popt_closedloop_treadmill"])
        pick["_sigma_rs"] = _sigma_rs_g2mult(pick[f"rsof_popt_closedloop_g2mult{SFX}"])
        examples[layer] = pick
        tag = "FORCED"
    else:
        sub = cand[cand.layer == layer]
        med = sub[g2d_rsq].median()
        pick = sub.iloc[(sub[g2d_rsq] - med).abs().argmin()]
        examples[layer] = pick
        tag = f"auto (median g2d R2={med:.3f}, n={len(sub)})"
    print(f"{layer}: {tag} -> {pick['session']} roi {int(pick['roi'])} "
          f"(Depth={pick[rsq_cols['gratio']]:.2f}, DepthxRS={pick[rsq_cols['g2mult']]:.2f}, "
          f"Conj={pick[g2d_rsq]:.2f}, depth_amp={pick['_depth_amp']:.2f} dff, "
          f"sigma_RS(g2mult)={pick['_sigma_rs']:.2f})")

# --- 2 x 4 figure (compact: shared axis labels, R^2 inside each panel) ---
fig, axes = plt.subplots(2, 4, figsize=(7.5, 5.0), sharex=True, sharey=True,
                         constrained_layout=True)
for irow, layer in enumerate(["Layer 2/3", "Layer 5"]):
    pick = examples[layer]
    session, roi = pick["session"], int(pick["roi"])
    trials_df = load_session_trials(sess_to_project[session], session)
    session_df = neurons_df[neurons_df.session == session]

    vmax, rs_edges, of_edges, nodata = plot_data_matrix(axes[irow, 0], trials_df, roi)
    axes[irow, 0].set_ylabel(layer)                 # row label = layer
    if irow == 0:
        axes[irow, 0].set_title("Measured")

    for icol, (model, label) in enumerate(fit_models, start=1):
        ax = axes[irow, icol]
        render_fit(ax, session_df, roi, model, vmax)
        overlay_nodata(ax, rs_edges, of_edges, nodata)
        if irow == 0:
            ax.set_title(label)
        r2 = pd.to_numeric(pd.Series([pick[rsq_cols[model]]]), errors="coerce").iloc[0]
        ax.text(0.05, 0.95, f"$R^2$={r2:.2f}", transform=ax.transAxes,
                ha="left", va="top", fontsize=8,
                bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.7))

    for icol in range(4):
        _style_axis(axes[irow, icol])

    # One colorbar per cell (row), to the right of the Conjunctive panel. All four panels
    # of a row share clim (vmin=0, vmax=vmax from the measured matrix, passed to every
    # render_fit), so a single bar describes the whole row.
    sm = plt.cm.ScalarMappable(norm=plt.Normalize(vmin=0, vmax=vmax), cmap="Reds")
    cbar = fig.colorbar(sm, ax=axes[irow, 3], fraction=0.046, pad=0.04)
    cbar.set_label("dF/F", fontsize=8)
    cbar.ax.tick_params(labelsize=7)

# Tick labels only on the outer edges: y on the left column, x on the bottom row.
for irow in range(2):
    for icol in range(4):
        if icol > 0:
            axes[irow, icol].set_yticklabels([])
        if irow == 0:
            axes[irow, icol].set_xticklabels([])

# Single shared axis labels, centred across the whole figure (once each).
fig.supxlabel("Running speed (cm/s)")
fig.supylabel("Optic flow (deg/s)")
fig.savefig("layer_fit_example_cells.pdf", bbox_inches="tight")
plt.show()
